# Clean up AnnDatapbject and add extended meta data 

So it is ready for further analysis

In [1]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

In [2]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data

# dataframes
import pandas as pd
import numpy as np

In [3]:
anndata_dir = str(here('data/anndata/'))
meta_dir = Path(here('data/metadata_harmonized'))

## Load

In [4]:
# Adata object
adata = ad.read_h5ad(os.path.join(anndata_dir, "AG_combined.h5ad"))

In [5]:
medication = pd.read_csv(os.path.join(meta_dir, 'meta_harmonized_medication.csv'))
patch = pd.read_csv(os.path.join(meta_dir, 'meta_harmonized_patch.csv'))

## Delete unused data

In [6]:
del adata.uns['leiden_1.5_colors']
del adata.uns['leiden_1.5_subcluster_colors']

## Add extended meta data

In [7]:
# Save the original index
orig_index = adata.obs.index

# Merge using the join keys, without touching the index
merged = (
    adata.obs
    .merge(medication, how="left", on=['name', 'sample', 'donor'])
    .merge(patch, how="left", on=['name', 'sample', 'donor'])
)

# Restore the original index and order
merged.index = orig_index

# Check index is the same order as the original Anndata
assert (adata.obs.index == orig_index).all(), "Index are not the same order"

# If they are the same order add to adata obs
adata.obs = merged

# Fill NA with "no" in diabetes_medication_harmonized
adata.obs['diabetes_medication_harmonized'] = adata.obs['diabetes_medication_harmonized'].fillna('no')

## Add annotation colors 

In [ ]:
# Add colors to AnnData categorical annotation
manual_anno_colors = {
    'acinar': '#c5b0d5',
    'acinar_reg_plus': '#FFCD01',
    'alpha': '#1f77b4',
    'beta': '#ff0000',
    'cycling': '#aec7e8',
    'delta': '#895129',
    'ductal': '#FF46A2',
    'ductal_mucin': '#F7B6D2',
    'endmt_early': '#ae0000',
    'endmt_late': '#44BDF8',
    'endothelial_islet': '#D844F8',
    'epsilon': '#BCF646',
    'gamma': '#FFBB78',
    'mast': '#0000FF',
    'myeloid': '#ee7733',
    'schwann': '#BCBD22',
    'stellate_activated': '#64F844',
    'stellate_quiescent': '#9467BD'
}

# ensure annotation is categorical
adata.obs["manual_annotation"] = adata.obs["manual_annotation"].astype("category")

# assign colors in the correct category order

adata.uns["manual_annotation_colors"] = [
    manual_anno_colors[c] for c in adata.obs["manual_annotation"].cat.categories
]

## Save object

In [8]:
adata.write(os.path.join(anndata_dir, 'AH_combined.h5ad')) 